# Phase 1: Exploratory Data Analysis (EDA) & User-Item Interaction Matrix

In this notebook, we explore our raw course recommendation dataset and build our first numerical representation: the **User-Item Interaction Matrix**.

In [ ]:
import pandas as pd
import numpy as np

# Load raw datasets
df_users = pd.read_csv('../data/raw/users.csv')
df_courses = pd.read_csv('../data/raw/courses.csv')
df_interactions = pd.read_csv('../data/raw/interactions.csv')

print(f"Users shape: {df_users.shape}")
print(f"Courses shape: {df_courses.shape}")
print(f"Interactions shape: {df_interactions.shape}")

## 1. Dataset Overview & Data Quality Check
Let's inspect missing values, duplicates, and column types.

In [ ]:
print("=== Missing Values in Interactions ===")
print(df_interactions.isnull().sum())

print("\n=== Duplicate Records in Interactions ===")
print(f"Duplicates count: {df_interactions.duplicated().sum()}")

print("\n=== Interaction Types Breakdown ===")
print(df_interactions['interaction_type'].value_counts())

### Insights on Data Quality & Signals:
- `rating` has missing values because only users who completed a course were prompted to leave a rating (explicit feedback).
- Non-rating interactions (VIEW, BOOKMARK, ENROLL, COMPLETE) represent implicit behavioral signals.
- The interaction counts strictly decrease along the funnel: VIEW (most common) > BOOKMARK > ENROLL > COMPLETE > RATE.

## 2. User & Course Interaction Distributions & Sparsity

In [ ]:
n_users = df_users['user_id'].nunique()
n_courses = df_courses['course_id'].nunique()
unique_user_course_pairs = df_interactions[['user_id', 'course_id']].drop_duplicates().shape[0]

possible_pairs = n_users * n_courses
sparsity = (1.0 - (unique_user_course_pairs / possible_pairs)) * 100

print(f"Total Unique Users: {n_users}")
print(f"Total Unique Courses: {n_courses}")
print(f"Unique User-Course Pairs Interacted: {unique_user_course_pairs}")
print(f"Total Possible Combinations: {possible_pairs}")
print(f"Interaction Matrix Sparsity: {sparsity:.2f}%")

### What does Sparsity tell us?
Sparsity measures the percentage of missing user-course relationships in our interaction matrix. High sparsity is typical in recommendation systems because learners only browse a small sample of the full catalog. ML algorithms (like Matrix Factorization) specifically excel at filling in these sparse blanks!

## 3. Converting Implicit Signals into Numerical Weights
We assign heuristic weights to each event type:
- `VIEW` = 1 point
- `BOOKMARK` = 2 points
- `ENROLL` = 3 points
- `COMPLETE` = 4 points

If a user has multiple actions on a course, we sum these weights to get a single consolidated interaction strength.

In [ ]:
weight_map = {
    'VIEW': 1,
    'BOOKMARK': 2,
    'ENROLL': 3,
    'COMPLETE': 4
}

# Exclude explicit RATE events for implicit score aggregation
df_implicit = df_interactions[df_interactions['interaction_type'] != 'RATE'].copy()
df_implicit['weight'] = df_implicit['interaction_type'].map(weight_map)

# Aggregate per (user_id, course_id)
interaction_summary = df_implicit.groupby(['user_id', 'course_id'])['weight'].sum().reset_index()

print("=== Sample Numerical Interaction Summary ===")
print(interaction_summary.head(10))

## 4. Constructing the User-Course Matrix
We pivot `interaction_summary` into a 2D matrix where:
- **Rows** = User IDs
- **Columns** = Course IDs
- **Values** = Cumulative interaction strength (0 if no interaction)

In [ ]:
user_item_matrix = interaction_summary.pivot(
    index='user_id',
    columns='course_id',
    values='weight'
).fillna(0)

print(f"User-Item Matrix Shape: {user_item_matrix.shape}")
user_item_matrix.iloc[:5, :8]